# Validate training-mean and ridge baselines

In [ ]:
"""Baseline validation robustness checks.

This notebook evaluates simple baseline models on the same model-ready payloads
used by the annual, winter, and summer XGBoost models. It writes a compact
summary table used to benchmark XGBoost against climatological-mean and RidgeCV
baselines under the same random, glacier-held-out, year-held-out and
O1Region-held-out split definitions.
"""

from __future__ import annotations

from pathlib import Path

import dill
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


from os import environ

configured_root = environ.get("WGMS_MB_ROOT")
ROOT = Path(configured_root).expanduser().resolve() if configured_root else Path.cwd().resolve()
if not configured_root:
    if (ROOT / "01_Dataset.ipynb").is_file():
        ROOT = ROOT.parent
    if not (ROOT / "code" / "01_Dataset.ipynb").is_file():
        raise RuntimeError("Start this notebook from the release root or its code directory, or set WGMS_MB_ROOT.")
elif not ROOT.is_dir():
    raise FileNotFoundError(f"The configured WGMS_MB_ROOT directory does not exist: {ROOT}")
OUT = ROOT / "results" / "analysis" / "baseline_validation_summary.csv"
RANDOM_STATE = 42
RIDGE_ALPHAS = np.logspace(-3, 3, 13)

PAYLOADS = {
    "annual": ROOT / "models" / "annual" / "preprocessed_data.pkl",
    "winter": ROOT / "models" / "annual_seasonal" / "preprocessed_winter_data.pkl",
    "summer": ROOT / "models" / "annual_seasonal" / "preprocessed_summer_data.pkl",
}

XGBOOST_SUMMARIES = {
    "annual": ROOT / "models" / "annual" / "xgboost_annual_validation_summary.csv",
    "winter": ROOT / "models" / "annual_seasonal" / "xgboost_winter_validation_summary.csv",
    "summer": ROOT / "models" / "annual_seasonal" / "xgboost_summer_validation_summary.csv",
}


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    yt = y_true[mask]
    yp = y_pred[mask]
    if len(yt) == 0:
        return {
            "n": 0,
            "R2": np.nan,
            "RMSE": np.nan,
            "MAE": np.nan,
            "Pearson_r": np.nan,
            "NRMSE_mean_abs": np.nan,
        }
    rmse = float(mean_squared_error(yt, yp) ** 0.5)
    mean_abs = float(np.mean(np.abs(yt)))
    return {
        "n": int(len(yt)),
        "R2": float(r2_score(yt, yp)) if len(yt) > 1 else np.nan,
        "RMSE": rmse,
        "MAE": float(mean_absolute_error(yt, yp)),
        "Pearson_r": float(np.corrcoef(yt, yp)[0, 1]) if len(yt) > 1 and np.std(yp) > 0 else np.nan,
        "NRMSE_mean_abs": rmse / mean_abs if mean_abs > 0 else np.nan,
    }


def load_payload(path: Path) -> dict:
    with path.open("rb") as handle:
        payload = dill.load(handle)
    return {
        "X": pd.DataFrame(np.asarray(payload["X_all"], dtype=float), columns=list(payload["feature_columns"])),
        "y": np.asarray(payload["y_all"], dtype=float),
        "rgiid": np.asarray(payload["rgiid_all"]),
        "year": np.asarray(payload["year_all"]),
        "o1region": np.asarray(payload.get("o1region_all", np.repeat(np.nan, len(payload["y_all"])))),
        "n_features": len(payload["feature_columns"]),
    }


def new_model(model_name: str):
    if model_name == "climatology_mean":
        return DummyRegressor(strategy="mean")
    if model_name == "ridge_cv":
        return make_pipeline(StandardScaler(), RidgeCV(alphas=RIDGE_ALPHAS))
    raise ValueError(f"Unknown model: {model_name}")


def fit_model(model, X, y):
    model.fit(X, y)
    return model


def evaluate_split(data: dict, model_name: str, split_name: str, splitter, groups=None) -> dict[str, float]:
    X = data["X"]
    y = data["y"]
    oof = np.full(len(y), np.nan, dtype=float)
    fold_count = 0
    for fold_count, (train_idx, test_idx) in enumerate(splitter.split(X, y, groups=groups), start=1):
        model = fit_model(new_model(model_name), X.iloc[train_idx], y[train_idx])
        oof[test_idx] = model.predict(X.iloc[test_idx])
    row = compute_metrics(y, oof)
    row.update({"model": model_name, "split": split_name, "folds": fold_count})
    return row


def xgboost_rows(target: str) -> list[dict]:
    path = XGBOOST_SUMMARIES[target]
    if not path.exists():
        return []
    df = pd.read_csv(path)
    rows = []
    for _, row in df.iterrows():
        split = str(row.get("split", ""))
        if split not in {"random_kfold", "logo_rgiid", "loyo_year"}:
            continue
        rows.append(
            {
                "target": target,
                "model": "xgboost",
                "split": split,
                "folds": np.nan,
                "n_features": np.nan,
                "n": int(row["n"]) if "n" in row and pd.notna(row["n"]) else np.nan,
                "R2": row["R2"],
                "RMSE": row["RMSE"],
                "MAE": row["MAE"],
                "Pearson_r": row["Pearson_r"],
                "NRMSE_mean_abs": row["NRMSE_mean_abs"],
            }
        )
    return rows


def main() -> None:
    rows = []
    for target, payload_path in PAYLOADS.items():
        data = load_payload(payload_path)
        for model_name in ["climatology_mean", "ridge_cv"]:
            splitters = [
                ("random_kfold", KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE), None),
                ("logo_rgiid", LeaveOneGroupOut(), data["rgiid"]),
                ("loyo_year", LeaveOneGroupOut(), data["year"]),
                ("logo_o1region", LeaveOneGroupOut(), data["o1region"]),
            ]
            for split_name, splitter, groups in splitters:
                row = evaluate_split(data, model_name, split_name, splitter, groups=groups)
                row.update({"target": target, "n_features": data["n_features"]})
                rows.append(row)
        rows.extend(xgboost_rows(target))

    out_df = pd.DataFrame(rows)
    out_df = out_df[
        [
            "target",
            "model",
            "split",
            "folds",
            "n_features",
            "n",
            "R2",
            "RMSE",
            "MAE",
            "Pearson_r",
            "NRMSE_mean_abs",
        ]
    ].sort_values(["target", "split", "model"])
    OUT.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(OUT, index=False)
    print(f"Wrote {OUT}")


if __name__ == "__main__":
    main()